In [1]:
# Cell 1: Environment Setup.
import os
os.environ["fix_mistral_regex"] = "True"
# os.environ["OMP_NUM_THREADS"] = "1"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # Extra 30% context lengths

# Install dependencies (run this if not already installed)
# !pip install unsloth vllm
# !pip install transformers==4.56.2
# !pip install --no-deps trl==0.22.2

In [2]:
# Set remote HF_TOKEN from local .env
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

# ssh -i ~/.ssh/id_ed25519 dataimaginations-heirarchical-reasoning@ssh.hf.space "echo 'export HF_TOKEN={hf_token}' >> ~/.bashrc"
print("✅ Token set! Restart remote shell to activate.")

✅ Token set! Restart remote shell to activate.


In [3]:
# Cell 2: HuggingFace Login
import os
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("✅ Logged in with HF_TOKEN")
else:
    login()
    print("✅ Logged in interactively")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Logged in with HF_TOKEN


In [4]:
"""
Two-Stage Curriculum SFT Training
=================================
Adapted from NVIDIA Nemotron methodology for QLoRA on RTX 4090.

Key Features:
1. Two-stage curriculum (4K → 8K context)
2. /think and /no_think mode control flags
3. Sample tracking for RL exclusion
4. Dual-mode training (thinking + direct response variants)

Usage:
    python scripts/sft_curriculum_trainer.py --stage 1
    python scripts/sft_curriculum_trainer.py --stage 2
"""

import os
import json
import argparse
import re
from datetime import datetime
from datasets import load_dataset, Dataset, concatenate_datasets
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()


True

In [ ]:

# ============================================================================
# CONFIGURATION
# ============================================================================


from unsloth import FastLanguageModel
import torch

# Hardware-adapted context lengths (RTX 4090)
STAGE_1_MAX_SEQ_LENGTH = 4096   # Foundation stage
STAGE_2_MAX_SEQ_LENGTH = 8192   # Extended reasoning stage

# Sample sizes per stage
STAGE_1_NEMOTRON_SAMPLES = 3000   # Math reasoning
STAGE_2_NEMOTRON_SAMPLES = 2000   # Additional harder problems
STAGE_2_CODE_SAMPLES = 1000       # Code reasoning (if available)

# LoRA configuration
LORA_RANK = 128
LORA_ALPHA = 128

# Training parameters
STAGE_1_EPOCHS = 1
STAGE_2_EPOCHS = 1
LEARNING_RATE = 2e-5  # Higher for SFT than RL
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4

# File paths
SAMPLE_TRACKING_FILE = "sft_used_samples.json"
STRATEGIC_GRAMS_FILE = "strategic_grams_deduplicated.json"


# Configuration
max_seq_length = 2048
lora_rank = 256      # <--- Bump this to 128 (Since you already did 64!)
lora_alpha = 256     # <--- Generally keep Alpha = Rank for Unsloth

print(f"⏳ Loading model with Rank {lora_rank}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/phi-4-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,
)

print("🔗 Attaching LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,        # <--- FIX: Use the variable, don"t hardcode!
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_alpha, # Set this to match the rank
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print(f"✅ Model loaded with Rank {lora_rank}!")

In [6]:

# ============================================================================
# SYSTEM PROMPTS WITH MODE CONTROL
# ============================================================================

# Thinking mode system prompt (detailed reasoning)
THINKING_SYSTEM_PROMPT = """You are an advanced reasoning assistant. When asked to think through a problem, show your complete reasoning process.

When you see /think in the user message, respond in this format:
<think>
[Show your step-by-step reasoning here]
</think>
<answer>
[Your final answer here]
</answer>"""

# Direct mode system prompt (no reasoning, fast response)
DIRECT_SYSTEM_PROMPT = """You are an advanced reasoning assistant. When asked for a direct answer, respond concisely without showing your work.

When you see /no_think in the user message, respond directly with just the answer - no thinking tags or explanation."""


In [7]:

# ============================================================================
# SAMPLE TRACKING
# ============================================================================

def load_used_samples():
    """Load the tracking file of samples used in SFT."""
    if os.path.exists(SAMPLE_TRACKING_FILE):
        with open(SAMPLE_TRACKING_FILE, "r") as f:
            return json.load(f)
    return {"stage_1": {}, "stage_2": {}}


def save_used_samples(tracking_data):
    """Save the tracking file."""
    with open(SAMPLE_TRACKING_FILE, "w") as f:
        json.dump(tracking_data, f, indent=2)
    print(f"💾 Saved sample tracking to '{SAMPLE_TRACKING_FILE}'")


def mark_samples_used(stage: str, source: str, sample_ids: list, tracking_data: dict):
    """Mark samples as used for a given stage and source."""
    if stage not in tracking_data:
        tracking_data[stage] = {}
    if source not in tracking_data[stage]:
        tracking_data[stage][source] = []
    tracking_data[stage][source].extend(sample_ids)
    return tracking_data


In [8]:

# ============================================================================
# DATA LOADING AND FORMATTING
# ============================================================================

def extract_thinking_and_answer(content: str) -> tuple[str, str]:
    """Extract thinking and answer portions from assistant response."""
    think_match = re.search(r'<think>(.*?)</think>', content, re.DOTALL)
    answer_match = re.search(r'<answer>(.*?)</answer>', content, re.DOTALL)
    
    thinking = think_match.group(1).strip() if think_match else ""
    answer = answer_match.group(1).strip() if answer_match else content.strip()
    
    return thinking, answer


def format_nemotron_for_sft(example, mode="thinking"):
    """
    Format Nemotron example for SFT training.
    
    Args:
        example: Raw Nemotron dataset example
        mode: "thinking" for <think> format, "direct" for no thinking
    
    Returns:
        Formatted example with messages and tracking info
    """
    messages = example.get('messages', [])
    
    user_content = ""
    assistant_content = ""
    
    for msg in messages:
        if msg['role'] == 'user':
            user_content = msg['content']
        elif msg['role'] == 'assistant':
            assistant_content = msg['content']
    
    if not user_content or not assistant_content:
        return None
    
    # Extract thinking and answer from original response
    thinking, answer = extract_thinking_and_answer(assistant_content)
    
    if mode == "thinking":
        # Add /think flag and format with thinking tags
        system_prompt = THINKING_SYSTEM_PROMPT
        user_message = f"/think {user_content}"
        
        if thinking:
            assistant_message = f"<think>\n{thinking}\n</think>\n<answer>\n{answer}\n</answer>"
        else:
            # If no explicit thinking, use the full content as thinking
            assistant_message = f"<think>\n{assistant_content}\n</think>\n<answer>\n{answer}\n</answer>"
    
    else:  # direct mode
        system_prompt = DIRECT_SYSTEM_PROMPT
        user_message = f"/no_think {user_content}"
        assistant_message = answer  # Just the answer, no tags
    
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message},
        ],
        "mode": mode,
        "source": "nemotron_math"
    }


def format_hicra_for_sft(example, mode="thinking"):
    """Format HICRA dataset example for SFT training."""
    user_content = example.get('prompt', '')
    expected_answer = str(example.get('answer', ''))
    
    if mode == "thinking":
        system_prompt = THINKING_SYSTEM_PROMPT
        user_message = f"/think {user_content}"
        # For HICRA, we construct a thinking response
        assistant_message = f"<think>\nLet me work through this problem step by step.\n</think>\n<answer>\n{expected_answer}\n</answer>"
    else:
        system_prompt = DIRECT_SYSTEM_PROMPT
        user_message = f"/no_think {user_content}"
        assistant_message = expected_answer
    
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message},
        ],
        "mode": mode,
        "source": "hicra"
    }


def load_stage_1_data(tracking_data: dict):
    """
    Load Stage 1 training data.
    
    Stage 1 focuses on:
    - Foundation with general + math reasoning
    - DUAL MODE: Both thinking and direct response variants
    - Shorter context (4K)
    """
    print("\n📚 Loading Stage 1 Data (Foundation)...")
    print("=" * 50)
    
    all_examples = []
    used_sample_ids = {"nemotron_math": [], "hicra": []}
    
    # 1. Load HICRA dataset (both modes)
    print("📂 Loading HICRA dataset...")
    try:
        hicra_dataset = load_dataset("json", data_files="reasoning_dataset_v2_train.json", split="train")
        
        for idx, example in enumerate(hicra_dataset):
            # Create BOTH thinking and direct variants (NVIDIA's parallel response strategy)
            thinking_example = format_hicra_for_sft(example, mode="thinking")
            direct_example = format_hicra_for_sft(example, mode="direct")
            
            if thinking_example:
                all_examples.append(thinking_example)
            if direct_example:
                all_examples.append(direct_example)
            
            used_sample_ids["hicra"].append(idx)
        
        print(f"   ✅ Loaded {len(hicra_dataset)} HICRA examples (×2 for dual mode)")
    except Exception as e:
        print(f"   ⚠️ Could not load HICRA: {e}")
    
    # 2. Load Nemotron Math (streaming)
    print(f"🌊 Streaming {STAGE_1_NEMOTRON_SAMPLES} Nemotron math examples...")
    try:
        nemotron_stream = load_dataset(
            "nvidia/Nemotron-Post-Training-Dataset-v1",
            split="math",
            streaming=True
        )
        
        count = 0
        for idx, example in enumerate(tqdm(nemotron_stream, total=STAGE_1_NEMOTRON_SAMPLES)):
            if count >= STAGE_1_NEMOTRON_SAMPLES:
                break
            
            # Stage 1: 50% thinking, 50% direct (parallel responses)
            if idx % 2 == 0:
                formatted = format_nemotron_for_sft(example, mode="thinking")
            else:
                formatted = format_nemotron_for_sft(example, mode="direct")
            
            if formatted:
                all_examples.append(formatted)
                used_sample_ids["nemotron_math"].append(idx)
                count += 1
        
        print(f"   ✅ Loaded {count} Nemotron examples")
    except Exception as e:
        print(f"   ⚠️ Could not load Nemotron: {e}")
    
    # Update tracking
    tracking_data = mark_samples_used("stage_1", "hicra", used_sample_ids["hicra"], tracking_data)
    tracking_data = mark_samples_used("stage_1", "nemotron_math", used_sample_ids["nemotron_math"], tracking_data)
    
    print(f"\n   📊 Total Stage 1 examples: {len(all_examples)}")
    print(f"   📊 Thinking mode: {sum(1 for e in all_examples if e.get('mode') == 'thinking')}")
    print(f"   📊 Direct mode: {sum(1 for e in all_examples if e.get('mode') == 'direct')}")
    
    return Dataset.from_list(all_examples), tracking_data


def load_stage_2_data(tracking_data: dict):
    """
    Load Stage 2 training data.
    
    Stage 2 focuses on:
    - Longer reasoning chains (8K context)
    - ALL examples in thinking mode
    - Harder problems + code reasoning
    """
    print("\n📚 Loading Stage 2 Data (Extended Reasoning)...")
    print("=" * 50)
    
    all_examples = []
    used_sample_ids = {"nemotron_math": [], "code": []}
    
    # Get previously used sample indices to avoid overlap
    stage_1_nemotron = set(tracking_data.get("stage_1", {}).get("nemotron_math", []))
    
    # 1. Load MORE Nemotron Math (harder problems, all thinking mode)
    print(f"🌊 Streaming {STAGE_2_NEMOTRON_SAMPLES} NEW Nemotron math examples...")
    try:
        nemotron_stream = load_dataset(
            "nvidia/Nemotron-Post-Training-Dataset-v1",
            split="math",
            streaming=True
        )
        
        # Skip Stage 1 samples
        start_idx = max(stage_1_nemotron) + 1 if stage_1_nemotron else 0
        
        count = 0
        for idx, example in enumerate(tqdm(nemotron_stream, total=start_idx + STAGE_2_NEMOTRON_SAMPLES)):
            if idx < start_idx:
                continue
            if count >= STAGE_2_NEMOTRON_SAMPLES:
                break
            
            # Stage 2: ALL thinking mode (as per NVIDIA paper)
            formatted = format_nemotron_for_sft(example, mode="thinking")
            
            if formatted:
                all_examples.append(formatted)
                used_sample_ids["nemotron_math"].append(idx)
                count += 1
        
        print(f"   ✅ Loaded {count} NEW Nemotron examples (thinking mode only)")
    except Exception as e:
        print(f"   ⚠️ Could not load Nemotron: {e}")
    
    # 2. Try to load Code reasoning (optional - may not be in your dataset)
    print(f"🔧 Attempting to load code reasoning data...")
    try:
        code_stream = load_dataset(
            "nvidia/Nemotron-Post-Training-Dataset-v1",
            split="code",
            streaming=True
        )
        
        count = 0
        for idx, example in enumerate(tqdm(code_stream, total=STAGE_2_CODE_SAMPLES)):
            if count >= STAGE_2_CODE_SAMPLES:
                break
            
            # Reuse format function (structure is similar)
            formatted = format_nemotron_for_sft(example, mode="thinking")
            
            if formatted:
                formatted["source"] = "code"
                all_examples.append(formatted)
                used_sample_ids["code"].append(idx)
                count += 1
        
        print(f"   ✅ Loaded {count} code examples")
    except Exception as e:
        print(f"   ⚠️ Could not load code split: {e}")
        print("   (This is optional - continuing with math only)")
    
    # Update tracking
    tracking_data = mark_samples_used("stage_2", "nemotron_math", used_sample_ids["nemotron_math"], tracking_data)
    tracking_data = mark_samples_used("stage_2", "code", used_sample_ids["code"], tracking_data)
    
    print(f"\n   📊 Total Stage 2 examples: {len(all_examples)}")
    
    return Dataset.from_list(all_examples), tracking_data


Tuning Tips:
- If you"re getting OOM: Lower NEMOTRON_SAMPLE_SIZE to 1000-2000
- If generations are too long: Lower MAX_ANSWER_TOKENS to 400
- If you want more data: Increase NEMOTRON_SAMPLE_SIZE to 5000+
The filtering keeps ~60-70% of examples typically, so 3000 samples → ~2000 usable examples mixed with your 729 HICRA examples.



**Optional: Use More GPU**
You could also try:

- `num_generations=6` (more diverse rollouts per step)
- Or increase `max_seq_length=1280` in cell 3 if Nemotron answers are very long

In [9]:

# ============================================================================
# TRAINING
# ============================================================================

def run_sft_training(dataset, stage: int, max_seq_length: int):
    """
    Run SFT training using Unsloth/TRL.
    """
    from unsloth import FastLanguageModel
    from trl import SFTTrainer, SFTConfig
    from huggingface_hub import login
    
    # Login to HuggingFace
    hf_token = os.getenv('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print("✅ Logged in with HF_TOKEN")
    
    output_dir = f"qwen3-14b-sft-stage{stage}"
    
    print(f"\n🚀 Starting Stage {stage} SFT Training")
    print("=" * 50)
    print(f"   Max sequence length: {max_seq_length}")
    print(f"   Training examples: {len(dataset)}")
    print(f"   Output directory: {output_dir}")
    
    # Load model
    print("\n⏳ Loading Qwen3-14B Base...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-14B-Base-bnb-4bit",
        max_seq_length=max_seq_length,
        load_in_4bit=True,
        fast_inference=False,
    )
    
    # For Stage 2, try to load Stage 1 checkpoint as starting point
    if stage == 2:
        stage_1_checkpoint = "qwen3-14b-sft-stage1"
        if os.path.exists(stage_1_checkpoint):
            print(f"📂 Loading Stage 1 checkpoint from {stage_1_checkpoint}...")
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, stage_1_checkpoint)
            model = model.merge_and_unload()
            print("   ✅ Merged Stage 1 adapters")
    
    # Attach LoRA
    print("🔗 Attaching LoRA adapters...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=LORA_ALPHA,
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    
    # Format function for the trainer
    def formatting_func(example):
        """Convert messages to chat format string."""
        messages = example["messages"]
        return tokenizer.apply_chat_template(messages, tokenize=False)
    
    # Training configuration
    training_args = SFTConfig(
        output_dir=output_dir,
        
        # Optimization
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        optim="paged_adamw_8bit",
        
        # Batching
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        
        # Duration
        num_train_epochs=STAGE_1_EPOCHS if stage == 1 else STAGE_2_EPOCHS,
        save_steps=200,
        logging_steps=10,
        
        # Memory
        fp16=False,
        bf16=True,
        
        # SFT specific
        max_seq_length=max_seq_length,
        packing=True,  # Efficient packing of short sequences
        
        # Logging
        report_to="tensorboard",
    )
    

Current settings `nvidia-smi` says: `6861MiB /  12282MiB`

In [11]:
from trl import SFTTrainer
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    formatting_func=formatting_func,
    args=training_args,
    )
    
# Train!
print("\n🏋️ Starting training...")
trainer.train()
    
# Save
print(f"\n💾 Saving model to {output_dir}...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
    
print(f"✅ Stage {stage} training complete!")
return model, tokenizer

Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


NameError: name 'model' is not defined

In [ ]:
# ============================================================================
# MAIN
# ============================================================================

def main():
    parser = argparse.ArgumentParser(description="Two-Stage Curriculum SFT Training")
    parser.add_argument("--stage", type=int, choices=[1, 2], required=True,
                        help="Training stage (1 = Foundation, 2 = Extended)")
    parser.add_argument("--dry-run", action="store_true",
                        help="Only load data, don't train (for testing)")
    args = parser.parse_args()
    
    print("=" * 60)
    print(f"  TWO-STAGE CURRICULUM SFT - STAGE {args.stage}")
    print("=" * 60)
    print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    
    # Load tracking data
    tracking_data = load_used_samples()
    
    # Load data for the appropriate stage
    if args.stage == 1:
        dataset, tracking_data = load_stage_1_data(tracking_data)
        max_seq_length = STAGE_1_MAX_SEQ_LENGTH
    else:
        dataset, tracking_data = load_stage_2_data(tracking_data)
        max_seq_length = STAGE_2_MAX_SEQ_LENGTH
    
    # Save updated tracking
    save_used_samples(tracking_data)
    
    # Shuffle the dataset
    dataset = dataset.shuffle(seed=42)
    
    # Show sample
    print("\n📝 Sample formatted example:")
    if len(dataset) > 0:
        sample = dataset[0]
        print(f"   Mode: {sample.get('mode', 'unknown')}")
        print(f"   Source: {sample.get('source', 'unknown')}")
        print(f"   Messages: {len(sample['messages'])} parts")
        for msg in sample['messages']:
            preview = msg['content'][:100].replace('\n', ' ')
            print(f"     [{msg['role']}]: {preview}...")
    
    if args.dry_run:
        print("\n⚠️ Dry run mode - skipping training")
        print(f"   Would train on {len(dataset)} examples")
        return
    
    # Run training
    run_sft_training(dataset, args.stage, max_seq_length)
    
    print("\n" + "=" * 60)
    print(f"  STAGE {args.stage} COMPLETE!")
    print("=" * 60)
    print(f"  Next steps:")
    if args.stage == 1:
        print("    1. Review Stage 1 checkpoint")
        print("    2. Run: python scripts/sft_curriculum_trainer.py --stage 2")
    else:
        print("    1. Review Stage 2 checkpoint")
        print("    2. Proceed to RL training (samples tracked in sft_used_samples.json)")
    print()


if __name__ == "__main__":
    main()

Restart your kernel, run cells 1-9, then run your resume training cell. 🚀



In [ ]:
# Cell 11: Run Training!
print("🏋️ Resuming training from checkpoint...")

# Option A: Resume from the latest checkpoint automatically
# trainer.train(resume_from_checkpoint=True)

# Option B: Resume from a specific checkpoint (if you want to go back in time)
trainer.train(resume_from_checkpoint="./phi-3.5-hicra-reasoner/checkpoint-500")

print("="*50)
print("✅ Training complete!")

In [ ]:
# Cell 9: Save Model
import os
# Option 1: Save locally
output_path = "Phi-3_5-reasoning-unsloth-HICRA-v1"
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print(f"✅ Model saved to {output_path}")
# Option 2: Push to HuggingFace Hub (uncomment to use)
repo_name = "DataImaginations/Phi-4-reasoning-HICRA-v1"
hf_token = os.getenv('HF_TOKEN')

# Option 2: Push to HuggingFace Hub (uncomment to use)
# repo_name = "DataImaginations/Llama-1B-Reasoning-v1"
# hf_token = os.getenv('HF_TOKEN')
# 
# print(f"⏳ Pushing to {repo_name}...")
# model.push_to_hub_merged(
#     repo_name,
#     tokenizer,
#     save_method="merged_16bit",
#     token=hf_token
# )
# print("✅ Model pushed to Hub!")

In [ ]:
# Cell: Merge LoRA adapters and save for evaluation
from unsloth import FastLanguageModel

# Load the adapter model
model, tokenizer = FastLanguageModel.from_pretrained(
    "Phi-3_5-reasoning-unsloth-HICRA-v1",
    max_seq_length=2048,
    load_in_4bit=True,
)

# Merge and save in 16-bit
print("⏳ Merging adapters...")
model.save_pretrained_merged(
    "Phi-3_5-reasoning-HICRA-v1-merged",  # New path for merged model
    tokenizer,
    save_method="merged_16bit",  # Full precision merged weights
)
print("✅ Merged model saved!")


### This creates a clean, standard HuggingFace model without any Unsloth-specific patches.


In [ ]:
from unsloth import FastLanguageModel

# Load the model with adapter (NOT merged yet)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Phi-3_5-reasoning-unsloth-HICRA-v1",  # Your adapter folder
    max_seq_length=2048,
    load_in_4bit=True,
)

# Use Unsloth's save method - this properly handles the merge!
model.save_pretrained_merged(
    "Phi-3_5-HICRA-MERGED-16bit",
    tokenizer,
    save_method="merged_16bit",  # This saves a clean HF-compatible model
)

print("✅ Saved merged 16-bit model!")

In [ ]:
from unsloth import FastLanguageModel

# Load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3.5-mini-instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

# Load your existing LoRA adapter directly
from peft import PeftModel
model = PeftModel.from_pretrained(model, "Phi-3_5-reasoning-unsloth-HICRA-v1")

# Merge adapter into base model
model = model.merge_and_unload()

# Save the merged model
model.save_pretrained("Phi-3_5-HICRA-MERGED")
tokenizer.save_pretrained("Phi-3_5-HICRA-MERGED")

print("✅ Merged model saved to Phi-3_5-HICRA-MERGED/")

## Test the Trained Model

In [ ]:
# Cell 11: Test Inference
from unsloth import FastLanguageModel

# Put model in inference mode
FastLanguageModel.for_inference(model)
# Create mask (1 for real tokens, 0 for padding) - mostly just all 1s for batch size 1
attention_mask = (inputs != tokenizer.pad_token_id).long()
# Test question
test_question = "A loan is repaid with 20 equal annual payments. The interest portion of the 16th payment is 400 and the interest portion of the 11th payment is 600. Find the interest portion of the 1st payment."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT.strip()},
    {"role": "user", "content": test_question}
]

# Tokenize with attention mask
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# Generate 4 attempts at once
outputs = model.generate(
    input_ids=inputs,
    attention_mask=attention_mask,
    max_new_tokens=2048,
    temperature=0.7,   # Keep some randomness for diversity
    do_sample=True,
    num_return_sequences=4, # <--- KEY CHANGE: Ask for 4 opinions
    pad_token_id=tokenizer.eos_token_id,
    stop_strings=["</answer>", "<|end|>", "<|endoftext|>"],
    tokenizer=tokenizer,
)

# Print all 4 to see if the "Smart" version is hiding in there
print("--- Voting ---")
for i, out in enumerate(outputs):
    # Decode skipping the prompt
    response = tokenizer.decode(out[inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nAttempt {i+1}:\n{response[:7048]}...") # Print start of answer
# Create attention mask (1 for all real tokens)
attention_mask = torch.ones_like(inputs)


# Actual Reasoning Solution from Claude

---

### Loan Amortization Problem
<think>
This is a loan amortization problem where we need to find the interest portion of the first payment given information about later payments.

Setting up the problem:


- Let A = the annual payment amount

- Let r = the interest rate per period

- The interest portion of payment k is: I_k = r × (balance before payment k)

Key insight: For loans with equal payments, the interest portions follow a pattern. Between consecutive payments:

I_{k+1} = I_k(1+r) - rA

This can be rearranged to show that (I_k - A) forms a geometric sequence:

I_k - A = (I_1 - A)(1+r)^{k-1}

Using the boundary condition: Since the loan is repaid in exactly 20 payments, after the 20th payment the balance is zero. This means:

I_{20} = Ar/(1+r)

Working through the algebra, we can derive:

I_k = A[1 - 1/(1+r)^{20-k+1}]

Applying known values:


- I_{11} = A[1 - 1/(1+r)^{10}] = 600

- I_{16} = A[1 - 1/(1+r)^{5}] = 400

Let x = (1+r)^5. Then:


- A(x² - 1)/x² = 600

- A(x - 1)/x = 400

From the second equation: A = 400x/(x-1)

Substituting into the first and simplifying:


- 400(x+1)/x = 600

- 400x + 400 = 600x

- x = 2

So (1+r)^5 = 2, and:

A = 400(2)/(2-1) = 800

Finally: I_1 = A[1 - 1/(1+r)^{20}] = 800[1 - 1/2^4] = 800(15/16) = 750

</think>

<answer>
The interest portion of the 1st payment is **750**.

To verify: With A = 800 and (1+r)^5 = 2:


- I_{11} = 800[1 - 1/4] = 600 ✓

- I_{16} = 800[1 - 1/2] = 400 ✓


</answer>

# Bench for unsloth_HICRA

In [ ]:
# benchmark cell
import lm_eval
from lm_eval.models.huggingface import HFLM

# 1. Load your local model
# If you just saved your model to "llama-1b-reasoning-final", point to that folder.
print("⏳ Loading model for evaluation...")

# We wrap the model in the Harness's HFLM wrapper
# 'pretrained' can be a local path OR a Hub ID (e.g., "david-barnes/my-model")
llm = HFLM(
    pretrained="Phi-3_5-HICRA-MERGED-16bit",  # Your merged model
    tokenizer="unsloth/Phi-3.5-mini-instruct-bnb-4bit",  # Add this line!
    batch_size=1,
    trust_remote_code=True,
    dtype="bfloat16"
)

# 2. Define the tasks you want
# These key names correspond to the harness registry.
# Note: "minerva_math" is often split by subject (algebra, etc), 
# so we usually run the main "math" group or specific subtasks.
task_list = [
    "arc_challenge",
    "hellaswag", 
    "winogrande",
    "piqa",
    "mmlu",
    "gsm8k",
    "truthfulqa_mc2",
]

print(f"🚀 Running BASELINE evaluation on: {task_list}...")
hicra_results = lm_eval.simple_evaluate(
    model=llm,
    tasks=task_list,
    num_fewshot=0,
    limit=100,  # Same limit for comparison
    log_samples=True,
)
# 4. Print a Pretty Table
from lm_eval.utils import make_table
print(make_table(hicra_results))

# 5. Save detailed results to JSON (Crucial for your blog!)
import json
with open("phi_3.5_hicra_reasoner_v1_benchmark_results.json", "w") as f:
    json.dump(hicra_results, f, indent=2)

# Soft VRAM clear

In [ ]:
import torch
import gc

# 1. Delete the Python variables holding the model
# (Wrap in try/except so it doesn't crash if they are already gone)
try:
    del model
    del tokenizer
    del trainer
except NameError:
    print("Variables already deleted or not defined.")

# 2. Python Garbage Collection (Clears CPU RAM)
gc.collect()

# 3. PyTorch Cache Clearing (The most important step for VRAM)
torch.cuda.empty_cache()

# Verify: Print current memory usage
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU Memory Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")